# Data Stewardship Toolkit — Google Colab runner

Run the whole stewardship pipeline interactively:

1. **Setup** — clones the toolkit and installs it.
2. **Checks** — verifies every dataset has a licence, metadata, checksum, provenance and template-matching headers.
3. **Templates** — generates a *headers-only CSV template* per dataset and downloads them, with a preview of exactly **which files to upload and with which columns**.
4. **Upload** — pick a dataset, click the upload button, choose your filled template; the file is validated against the header template and ingested as a **new dataset version** (checksum + provenance recorded).
5. **Dashboard** — re-verifies everything and renders the HTML stewardship dashboard inline.

*Menu: Runtime → Run all, or run the cells one by one. Outside Colab the
cells degrade gracefully to plain Jupyter / terminal instructions.*

In [ ]:
#@title 1 · Setup — clone the toolkit and install it { display-mode: "form" }
REPO_URL = "https://github.com/YOUR-USERNAME/data-stewardship-toolkit.git" #@param {type:"string"}

import subprocess, sys
from pathlib import Path

# Inside an already-cloned repo? use it (e.g. running locally)
cwd = Path.cwd()
if (cwd / "src" / "data_stewardship").is_dir():
    ROOT = cwd
elif Path("/content").exists():                      # Google Colab
    ROOT = Path("/content/data-stewardship-toolkit")
else:                                                # local clone
    ROOT = (cwd / "data-stewardship-toolkit").resolve()

if not ROOT.exists():
    print(f"Cloning {REPO_URL} ...")
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            f"Clone failed:\n{r.stderr}\n"
            f"Fix REPO_URL above (must point to your pushed repository), or upload "
            f"the toolkit ZIP to Colab and unzip it to {ROOT}."
        )

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[dev]"],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit(f"Install failed:\n{r.stderr}")

sys.path.insert(0, str(ROOT / "scripts"))
import colab_run  # noqa: E402

print("Toolkit ready at:", ROOT)
print("Google Colab detected:", colab_run.IN_COLAB)

In [ ]:
#@title 2 · Run all stewardship checks { display-mode: "form" }
checks_frame = colab_run.run_checks(ROOT)
checks_frame

In [ ]:
#@title 3 · Generate templates, preview what to upload & download them { display-mode: "form" }
DOWNLOAD_TEMPLATES = True #@param {type:"boolean"}

templates = colab_run.make_templates(ROOT)

print()
preview = colab_run.preview_uploads(ROOT)
display(preview.style.set_caption("Files to upload - one row per dataset"))

print()
colab_run.display_template_headers(ROOT)

if DOWNLOAD_TEMPLATES:
    for t in templates:
        colab_run.download_template(ROOT, t.name.replace("_template.csv", ""))

In [ ]:
#@title 4 · Upload filled data (upload widget) { display-mode: "form" }
ACTOR = "colab-user" #@param {type:"string"}

import ipywidgets as widgets
from data_stewardship.checks import read_inventory

dataset_ids = [r["dataset_id"] for r in read_inventory(ROOT)]

dataset_dd = widgets.Dropdown(options=dataset_ids, description="Dataset:",
                              style={"description_width": "initial"})
bump_dd = widgets.Dropdown(options=["patch", "minor", "major"], value="minor",
                           description="Version bump:",
                           style={"description_width": "initial"})
actor_txt = widgets.Text(value=ACTOR, description="Your name:",
                         style={"description_width": "initial"})
upload_btn = widgets.Button(description="Upload file for selected dataset",
                            button_style="primary", icon="upload",
                            tooltip="Opens the file chooser; the CSV header is validated against the dataset template before ingestion.")
out = widgets.Output()

def _on_upload(_):
    out.clear_output()
    with out:
        colab_run.upload_and_ingest(ROOT, dataset_dd.value,
                                    actor_txt.value.strip() or "colab-user",
                                    bump_dd.value)

upload_btn.on_click(_on_upload)
display(dataset_dd, bump_dd, actor_txt, upload_btn, out)

In [ ]:
#@title 5 · Re-verify & show the stewardship dashboard { display-mode: "form" }
SHOW_DASHBOARD = True #@param {type:"boolean"}
DOWNLOAD_DASHBOARD = False #@param {type:"boolean"}

if SHOW_DASHBOARD:
    colab_run.show_dashboard(ROOT, download=DOWNLOAD_DASHBOARD)

## 6 · Publish the new version back to GitHub (optional)

Colab cannot push without credentials. Two supported options:

**Option A — download the updated repo and push from your machine**

```python
!cd /content && zip -rq stewardship-updated.zip data-stewardship-toolkit -x "*__pycache__*" "*.egg-info*"
from google.colab import files; files.download("/content/stewardship-updated.zip")
```
Then, on your machine: unzip, `cd data-stewardship-toolkit`, commit and push.
Every ingestion added a provenance event and bumped the version, so CI will
re-verify the full chain of custody.

**Option B — push directly from Colab with a token**

```python
TOKEN = ""  # GitHub personal access token (never commit it!)
!cd /content/data-stewardship-toolkit && git config user.name "Colab User" && git config user.email "{YOUR-EMAIL}"
!cd /content/data-stewardship-toolkit && git add -A && git commit -m "Ingest new data via Colab" && git push https://<TOKEN>@github.com/YOUR-USERNAME/data-stewardship-toolkit.git main
```

---

*Toolkit code: MIT · demo datasets: CC0-1.0 · generated files land in
`datasets/<id>/data/`, `checksums/sha256sums.txt` and
`datasets/<id>/provenance.jsonl`.*